# From a pitch video to an offer prediction

This notebook follows **Firefighter1** from video annotations to measured concepts and a prediction of whether any investor makes an offer. It then examines the annotations, human agreement, and predictive performance across **100 episodes**.

**Start here:** open the notebook from the repository root, select the project's Python environment, then **Restart Kernel and Run All**. The demo uses the included video and saved annotation records; no API key is needed. See the [setup instructions](README.md#run-the-demo) to install the required packages.

1. [Annotating a single episode](#annotating-the-pitch): view the video, codebook, prompt, annotation record, and timeline.
2. [Validate with a human coder's annotations](#human-validation): inspect the completed Firefighter1 coding sheet and compare the human and Gemini annotations.
3. [From annotations to concepts and an offer prediction](#measuring-and-predicting): trace an answer-family concept, inspect all 46 model concepts, and compare the prediction from a model trained on the other 99 episodes with Firefighter1's outcome.
4. [Across the full 100 episodes](#across-the-100-videos): explore summary statistics and concept distributions, review human validation on 10 episodes, and compare predictive performance across concept families.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline").is_dir():
    raise RuntimeError("Open this notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML, Video
from analysis import demo, panel_explorer as px
from analysis.panel_features import history_table, FEATURES
from analysis.panel_bottleneck import ladder_columns
from analysis.proposal_figures import figure2, chance_table

%matplotlib inline
pd.set_option("display.max_colwidth", 110)

ANNOTATION_MODE = "saved"
EXECUTE_LIVE = False
RECOMPUTE_ANALYSIS = False
VIDEO_PATH = "data/shark tank/videos/Can The Sharks Add Value To Firefighter1_ _ Shark Tank US _ Shark Tank Global.mp4" # Optional local copy of the Firefighter1 video.
REFERENCE_RUN = demo.REFERENCE
EXAMPLE_VIDEO_ID = demo.EXAMPLE_ID
review_info = demo.review_manifest()

# Load the saved 100-episode research run; this makes no Gemini API calls.
# All paths below are relative to the repository root:
#   runs/full-corpus-100/<video_id>/results/{primary,replicate}.json
#     contains the per-episode annotation records. The analysis selects one pass
#     per episode; its video_id and source are recorded in analysis/all100/features.csv.
#   runs/full-corpus-100/analysis/all100/ contains the saved analysis tables.
# Returned objects:
#   data: full annotation tables (e.g. data.videos, data.turns, data.decisions,
#         data.prepared, data.stage_a), assembled from the selected JSON records.
#   features: episode-level model inputs and outcomes, regenerated from data and
#             checked against analysis/all100/features.csv.
#   saved: DataFrames read from features.csv, metrics.csv, deltas.csv,
#          predictions.csv, and single_feature.csv in analysis/all100/.
#          For example, saved["metrics"] supplies the Figure 2 performance table.
#   manifest: analysis/all100/manifest.json, documenting the episode IDs,
#             analysis settings, and input-file hashes.
# Loader and verification logic: analysis/demo.py, load_reference().
data, features, saved, manifest = demo.load_reference()
vdir, result, instrument, source, _ = demo.encounter(REFERENCE_RUN, EXAMPLE_VIDEO_ID, review_info["pass"])
display(Markdown(f"Loaded **{len(features)} pitches**, including Firefighter1."))

<a id="annotating-the-pitch"></a>
## 1. Annotating a single episode

If the Firefighter1 MP4 is available locally, the player below uses that file without storing the video inside the notebook. Anonymous downloads may omit the MP4. Use the YouTube button below if the video is unavailable or playback fails.

Gemini-3.1-pro received the codebooks and annotated this full episode twice so we can check internal consistency. Below we inspect the archived instrument and saved records. 

Source: [instrument and record inspection](analysis/panel_explorer.py), [two-pass annotation pipeline](pipeline/stage_b/panel_pipeline.py).

In [ ]:
video_file = Path(VIDEO_PATH).expanduser() if VIDEO_PATH else None
video_available = video_file is not None and video_file.is_file()
if video_available:
    with video_file.open("rb") as video_stream:
        video_available = not video_stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1")
if video_available:
    display(Video(filename=str(video_file), embed=False, width=760))
display(HTML(f'<p style="padding:24px;border:1px solid #ccd5dd;border-radius:8px;max-width:710px">'
             f'<a href="{demo.SOURCE_URL}" target="_blank" rel="noopener" style="font-size:20px">'
             '▶ Watch Firefighter1 on YouTube</a><br>Opens the original video in your browser.</p>'))


### Codebook and Prompt

In [ ]:
book_table = px.codebook_table(instrument.books.panel)
display(book_table.head(8))
# The full prompt is available as instrument.prompt_text.
print(px.prompt_excerpt(instrument, head=3000, tail=250))

### Inspect the annotation record returned by Gemini

The table below shows annotation record returned by Gemini on **Firefighter1**, structured into a table and visualized into a timeline (as used in Fig1)

Time marks the start of each turn. The compact codes show question severity, answer substance and coherence, checkable specifics (✓), and decision outcomes. The frame is derived from the annotated concept. Delivery and preparedness ratings belong to the prepared pitch; the detailed record views follow in Section 2.

If you wish to inspect the original json file returned by Gemini,  open the dropdown at the bottom of this table.

Source: [Saved, parsed annotation record](runs/full-corpus-100/st_8a6fd25840/results/primary.json).


In [ ]:
annotation_record = demo.annotation_record(result, instrument.books)
display(Markdown(f"**{EXAMPLE_VIDEO_ID} · {source} pass · {len(annotation_record)} turns**"))
record_style = (annotation_record.style
    .hide(axis="index")
    .format(escape="html")
    .set_uuid("annotation_record")
    .set_table_styles([
        {"selector": "", "props": [("width", "100%"), ("border-collapse", "collapse")]},
        {"selector": "th", "props": [("background-color", "#eeebfa"), ("color", "#7159bd"),
                                     ("text-align", "left"), ("padding", "8px 12px"),
                                     ("position", "sticky"), ("top", "0")]},
        {"selector": "td", "props": [("text-align", "left"), ("padding", "6px 12px"),
                                     ("color", "#333"), ("vertical-align", "top")]},
        {"selector": "tbody tr:nth-child(odd)", "props": [("background-color", "#fff")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#f7f7f8")]},
    ]))
display(HTML('<div style="max-height:560px;overflow:auto;border:1px solid #ddd;'
             'border-radius:8px">' + record_style.to_html() + '</div>'))

# Prefer Gemini's untouched response; shared checkouts may only contain parsed results.
from html import escape
raw_json_path = vdir / "raw" / f"{source}.json"
annotation_json_path = (raw_json_path if raw_json_path.is_file()
                        else vdir / "results" / px.PASS_FILES[source])
json_label = ("Saved, parsed annotation record")
annotation_json_text = annotation_json_path.read_text(encoding="utf-8")
display(HTML('<details><summary>Inspect annotation JSON</summary>'
             f'<p>{escape(json_label)}</p>'
             '<pre style="max-height:560px;overflow:auto;white-space:pre;'
             'padding:12px;border:1px solid #ddd;border-radius:8px">'
             + escape(annotation_json_text) + '</pre></details>'))


**Visualize the record as a timeline and annotated close-up (Fig 1)**

The dashed box identifies the **2:05–3:52** window enlarged below: shaded bands group exchanges, questions show their concept and severity, and answers show substance ratings and checkable support (✓). Arrows connect answers to the questions they respond to. 

Source: [panel explorer plotting functions](analysis/panel_explorer.py).

In [ ]:
fig = px.record_timeline_figure(
    result, start_s=125, end_s=232,
    title=f"Firefighter1 · timeline and annotations")
display(fig)
plt.close(fig)

<a id="human-validation"></a>
## 2. Validate with a human coder's annotations

The next cell embeds a read-only preview of the coding sheet completed by a human coder for **Firefighter1**.

Download the [completed coding sheet (HTML)](examples/firefighter1_review/pages/human1__completed.html) and open the downloaded file in your browser to see the full human annotation interface and coding.

The next cell calculates agreement using the project's current reporting rules. The table shows exact agreement, within-one agreement for ordered scales (1-5), Krippendorff's α, and Gwet's AC1 (categorical/binary) or AC2 (ordinal). A dominant category means at least 75% of the combined ratings share one value; read AC1/AC2 alongside α for these rows. A dash means the statistic is undefined or does not apply. 

Sources: [sheet import](scripts/review_panel.py) → [per-kind reporting](pipeline/stage_b/review_apply.py) → [agreement statistics](pipeline/reliability/agreement.py). The scales and reporting rules come from [validation_reporting.yaml](pipeline/codebooks/validation_reporting.yaml); [demo assembly](analysis/demo.py) selects the columns to display.

In [ ]:

OPEN_REVIEW_SHEET = "completed"
if OPEN_REVIEW_SHEET not in (None, "completed", "blank"):
    raise ValueError('Choose None, "completed", or "blank".')
if OPEN_REVIEW_SHEET:
    demo.open_review_sheet(completed=OPEN_REVIEW_SHEET == "completed")

human = demo.load_human_review()
if human is None:
    display(Markdown("**Awaiting the author's annotations.** [Open the coding sheet](examples/firefighter1_review/pages/human1__tasks.html), then use **Download all sheets** after coding."))
else:
    display(HTML(demo.review_sheet_preview()))
    display(demo.human_agreement_table(human))
    clock_row = human["cards"].set_index("path").loc["pitch:first_decision"]
    clock_difference = human["pooled"]["reporting"].get("cut", {}).get("figures", {}).get("mean_signed", {}).get("value")
    decision = human["decision_turn"]
    
    presentation = human["cards"].loc[human["cards"]["class"].isin(["stage_a", "prep_pres", "venture"]), ["code", "human1", "model"]]
    display(HTML("<details><summary>Inspect the seven presentation and investment-ask comparisons</summary>" + presentation.to_html(index=False) + "</details>"))
    display(Markdown("[All item comparisons and notes](examples/firefighter1_review/derived/item_comparison.csv) · [Detailed agreement output](examples/firefighter1_review/derived/reporting.json)"))

<a id="measuring-and-predicting"></a>
## 3. The concept bottleneck model: from annotations to concepts that predict the funding decision

### From answer annotations to a concept

We walk through the process with one family of concepts: **founder-answer**. It describes how entrepreneurs respond to investors' questions using six annotation codes: **RS-RESP** (directness: addresses, partially addresses, or deflects the question), **RS-EVID** (whether the answer gives checkable support), **RS-UNCERT** (whether it admits uncertainty or concedes a point), and **PREP-SUBS**, **PREP-COH**, and **PREP-CTX** (1–5 ratings of substance, coherence, and contextualisation). For each code, the model calculate the **mean** of the ratings or **shares across eligible answers**. Two further concepts measure substance and checkable support specifically in answers linked, through `responds_to`, to hard investor questions (**PR-SEV = 2 or 3**). Together these make eight concepts in the answer family.

Eligible interactions that go into the concept bottleneck model occur strictly **before the first investor decision of any kind**, including an exit or an offer. The table below lists the eligible answers, their support ratings, and their summaries so the reported share can be checked against the record. For Firefighter1, **7 of 12 rated answers contain checkable support: 7 ÷ 12 = 58.3%**.

Sources: [answer and question code definitions](pipeline/codebooks/stage_b_panel.md), [annotation-to-number conversion and eligibility rules](analysis/panel_dataset.py), and [feature construction and history table](analysis/panel_features.py).


In [ ]:
example = features.set_index("video_id").loc[EXAMPLE_VIDEO_ID]
display(demo.support_trace(data, EXAMPLE_VIDEO_ID, result))
display(Markdown(f"Checkable support: **{example.a_evid_share:.1%}** of **{int(example.a_evid_share_n)}** eligible rated answers."))


### From individual annotations to the full set of model concepts

Using annotations from Firefighter1, the table below shows each concept, its'family, source annotations, and the calculation to turn the annotation into a quantitative concept. Values retain their original units before model standardisation; shares range from 0 to 1.

All interaction measures use the eligible history defined above. Means and shares exclude missing ratings; with no observations they remain missing unless noted. Reaction and next-move measures skip social, demonstration, and `NONE` turns.

Sources: [concept definitions and aggregation](analysis/panel_features.py), [reaction windows](analysis/reaction_events.py), and [model preprocessing](analysis/panel_bottleneck.py).


In [ ]:
concept_columns = [c for c in ladder_columns(features, tuple(manifest["models"]["C5"])) if not c.endswith("_missing")]

# Source annotations and concise calculations; shared rules are stated above the table.
concept_details = {'ask_log_amount': ('VC-ASK: ask_amount_usd', 'log₁₀(ask in USD); missing if absent or ≤0.'),
 'ask_equity_pct': ('VC-ASK: ask_equity_pct', 'Equity requested (%).'),
 'hist_turns': ('TURN-FUNC; turn order; DEC-KIND; EX-KIND', 'Number of eligible turns.'),
 'hist_seconds': ('Turn start_s and end_s; eligibility annotations',
                  'Sum of turn durations in seconds: Σ max(0, end − start).'),
 'hist_questions': ('TURN-FUNC = question; actor role', 'Number of investor questions.'),
 'sa_rank_pitch_positivity': ('Stage A pitch_positivity batch rankings',
                              'Plackett–Luce score from batch rankings, centred at zero.'),
 'pres_evid': ('PRES-EVID', 'Count distinct supported claims; exclude the investment ask.'),
 'q_askers': ('TURN-FUNC = question; speaker ID and actor role',
              'Number of distinct investors asking questions.'),
 'q_sev_mean': ('PR-SEV on investor questions', 'Mean question severity (1–3).'),
 'q_hard_share': ('PR-SEV on investor questions', 'Questions rated 2–3 ÷ severity-rated questions.'),
 'q_sev_trend': ('PR-SEV; investor-question turn order',
                 'Later-half mean severity − earlier-half mean; omit the odd middle rating. Fewer than two '
                 'ratings: 0; none: flag missing.'),
 'q_pursuit_share': ('PR-SEV; speaker ID; investor-question turn order',
                     'Same-investor, rising-severity pairs ÷ consecutive question pairs with both severities '
                     'rated.'),
 'q_frame_switch_share': ('PR-CONCEPT → derived frame; investor-question turn order',
                          'Frame-changing pairs ÷ consecutive question pairs with both frames '
                          'promotion/prevention.'),
 'q_concept_entropy': ('PR-CONCEPT on investor questions',
                       'Concept diversity: −Σ p log₂(p); include deal_terms/other, exclude NONE. No '
                       'concepts: missing.'),
 'a_resp_mean': ('RS-RESP on eligible answers',
                 'Mean of addresses = 1, partial = 0.5, deflects = 0; exclude NONE.'),
 'a_evid_share': ('RS-EVID on eligible answers', 'Answers with checkable support ÷ rated answers.'),
 'a_uncert_share': ('RS-UNCERT on eligible answers',
                    'Answers admitting uncertainty or conceding a point ÷ rated answers.'),
 'a_subs_hard_mean': ('PREP-SUBS; responds_to; PR-SEV',
                      'Mean substance (1–5) for answers to severity 2–3 questions; count each answer once.'),
 'a_evid_hard_share': ('RS-EVID; responds_to; PR-SEV',
                       'Supported answers ÷ rated answers to severity 2–3 questions; count each answer '
                       'once.'),
 'rx_reaction_rate': ('TURN-FUNC; responds_to; speaker roles; turn order',
                      'Mean reaction count per linked answer, including zero-reaction answers.'),
 'founder_gender': ('VC-GENDER', 'Category → 0/1 indicators; levels in <5 training pitches are pooled.'),
 'primary_industry': ('VC-INDUSTRY', 'Category → 0/1 indicators; levels in <5 training pitches are pooled.'),
 'product_form': ('VC-FORM', 'Category → 0/1 indicators; levels in <5 training pitches are pooled.'),
 'customer_type': ('VC-CUSTOMER', 'Category → 0/1 indicators; levels in <5 training pitches are pooled.'),
 'pres_subs': ('PRES-SUBS', 'Prepared-pitch rating (1–5).'),
 'pres_coh': ('PRES-COH', 'Prepared-pitch rating (1–5).'),
 'pres_ctx': ('PRES-CTX', 'Prepared-pitch rating (1–5).'),
 'a_subs_mean': ('PREP-SUBS on eligible answers', 'Mean answer rating (1–5).'),
 'a_coh_mean': ('PREP-COH on eligible answers', 'Mean answer rating (1–5).'),
 'a_ctx_mean': ('PREP-CTX on eligible answers', 'Mean answer rating (1–5).'),
 'q_frame_promotion_share': ('PR-CONCEPT → codebook-derived frame',
                             'Promotion-framed questions ÷ all investor questions.'),
 'q_frame_prevention_share': ('PR-CONCEPT → codebook-derived frame',
                              'Prevention-framed questions ÷ all investor questions.'),
 'q_deal_terms_share': ('PR-CONCEPT on investor questions',
                        'Questions coded deal_terms ÷ all investor questions.'),
 'q_other_share': ('PR-CONCEPT on investor questions', 'Questions coded other ÷ all investor questions.'),
 'rx_receipt_share': ('TURN-FUNC = accepts; responds_to; speaker roles; turn order',
                      'Answers with ≥1 accepts reaction in their window ÷ linked answers.'),
 'nm_receipt_share': ("TURN-FUNC = accepts; responds_to; original questioner's speaker ID; turn order",
                      'Links whose next investor move is the questioner’s accepts turn ÷ all answer–question '
                      'links, including those with no next move.'),
 'rx_followup_share': ('TURN-FUNC = question; responds_to; speaker roles; turn order',
                       'Answers with ≥1 question reaction in their window ÷ linked answers.'),
 'nm_followup_share': ("TURN-FUNC = question; responds_to; original questioner's speaker ID; turn order",
                       'Links whose next investor move is the questioner’s question turn ÷ all '
                       'answer–question links, including those with no next move.'),
 'rx_challenge_share': ('TURN-FUNC = challenges; responds_to; speaker roles; turn order',
                        'Answers with ≥1 challenges reaction in their window ÷ linked answers.'),
 'nm_challenge_share': ("TURN-FUNC = challenges; responds_to; original questioner's speaker ID; turn order",
                        'Links whose next investor move is the questioner’s challenges turn ÷ all '
                        'answer–question links, including those with no next move.'),
 'accepts_turns': ('TURN-FUNC = accepts; actor role = investor', 'Number of investor turns coded accepts.'),
 'challenge_turns': ('TURN-FUNC = challenges; actor role = investor',
                     'Number of investor turns coded challenges.'),
 'social_turns': ('TURN-FUNC = social; actor role = investor', 'Number of investor turns coded social.'),
 'founder_social': ('TURN-FUNC = social; actor role = founder', 'Number of founder turns coded social.'),
 'hostile_at_founder': ('TURN-FUNC = hostile; actor role = investor',
                        'Number of investor turns coded hostile.'),
 'founder_repairs': ('TURN-FUNC = repair; actor role = founder', 'Number of founder turns coded repair.')}

family_labels = {
    "control": "Venture and ask", "exposure": "Interaction amount", "stage_a": "Pitch delivery",
    "prepared": "Pitch preparedness", "questions": "Investor questions", "answers": "Founder answers",
    "reactions": "Investor reactions", "rapport_repair": "Rapport and repair",
}
assert set(concept_columns) <= concept_details.keys(), "A model concept needs its annotation and calculation documented."
concepts = pd.DataFrame([
    {"Concept": FEATURES[name].get("label", name),
     "Family": family_labels[FEATURES[name]["family"]],
     "Value": example.get(name),
     "Annotations used": concept_details[name][0],
     "How the number is calculated": concept_details[name][1]}
    for name in concept_columns
])
display(Markdown(f"**{len(concept_columns)} named concepts** enter the full model specification; missingness indicators and category encodings are separate."))
concept_style = (concepts.style.hide(axis="index")
    .format(precision=3, na_rep="—", escape="html")
    .set_uuid("concept_definitions")
    .set_table_styles([
        {"selector": "", "props": [("width", "100%"), ("border-collapse", "collapse")]},
        {"selector": "th", "props": [("text-align", "left"), ("background-color", "#eeebfa"),
                                     ("color", "#7159bd"), ("position", "sticky"), ("top", "0")]},
        {"selector": "th, td", "props": [("padding", "8px 12px"), ("vertical-align", "top"),
                                         ("white-space", "normal"), ("text-align", "left")]},
        {"selector": "td.col3, td.col4", "props": [("min-width", "220px")]},
    ]))
display(HTML('<div style="max-height:720px;overflow:auto;border:1px solid #ddd;border-radius:8px">'
             + concept_style.to_html() + '</div>'))
print("Full corpus concept table:", features.shape)
display(features.head(3))


### What does the model predict and is it right?

The concept bottleneck model is the project's ridge logistic regression. It is trained on the **other 99 videos**, using the reported preprocessing and penalty-selection procedure. The output is the probability that **any investor makes an offer**, not the probability of accepting or closing a deal.

The comparison below uses **50% as the classification threshold**: a probability at or above 50% predicts an offer. It compares that prediction with the recorded episode outcome (`any_offer`) and shows the offer and acceptance events from the saved annotations. Correctness here concerns whether an offer occurred; acceptance is shown for context.

The contribution table is on the log-odds scale: positive values raise this fitted prediction, and negative values lower it. These are model contributions, not causal effects. One example's probability does not establish predictive accuracy or calibration.

Source: [model fitting and prediction breakdown](analysis/panel_bottleneck.py), [one-fold assembly](analysis/demo.py).

In [ ]:
probability, contributions, fitted = demo.held_out_example(features, manifest, EXAMPLE_VIDEO_ID)
expected = saved["predictions"].set_index("video_id").loc[EXAMPLE_VIDEO_ID, "p_C5"]
assert abs(probability - expected) < 1e-6
assert EXAMPLE_VIDEO_ID not in fitted["folds"][0]["train_groups"]
display(Markdown(f"### Predicted probability of an offer: {probability:.1%}\n\nTraining pitches: {len(fitted['folds'][0]['train_groups'])}. "))
# Compare the prediction with the episode-level outcome used by the analysis.
PREDICTION_THRESHOLD = 0.50
actual_offer = features.set_index("video_id").loc[EXAMPLE_VIDEO_ID, "any_offer"]
predicted_offer = probability >= PREDICTION_THRESHOLD
actual_known = pd.notna(actual_offer)
comparison = pd.DataFrame([{
    "Predicted probability": f"{probability:.1%}",
    "Prediction (50% threshold)": "Offer" if predicted_offer else "No offer",
    "Actual episode outcome": ("Offer" if actual_offer == 1 else "No offer") if actual_known else "Unknown",
    "Is the prediction correct?": ("Yes" if predicted_offer == bool(actual_offer) else "No") if actual_known else "Cannot assess",
}])
comparison_style = (comparison.style
    .hide(axis="index")
    .format(escape="html")
    .set_uuid("prediction_comparison")
    .set_properties(subset=comparison.columns[:-1], **{
        "background-color": "#dcfce7", "color": "#166534", "font-weight": "700",
    })
    .map(lambda value: (
        "background-color:#dcfce7;color:#166534;font-weight:700" if value == "Yes" else
        "background-color:#fee2e2;color:#991b1b;font-weight:700" if value == "No" else
        "background-color:#f3f4f6;color:#4b5563;font-weight:700"
    ), subset=["Is the prediction correct?"])
    .set_table_styles([
        {"selector": "th, td", "props": [("padding", "10px 14px"), ("text-align", "left")]},
    ]))
display(HTML(comparison_style.to_html()))

# Show the recorded events that explain the observed outcome.
episode_decisions = px.decisions_table(result)
outcome_events = episode_decisions.loc[
    episode_decisions["Kind"].isin(["offer", "joins_offer", "founder_accepts"]),
    ["Clock", "Who", "Kind", "Terms"],
].rename(columns={"Clock": "Time", "Kind": "Episode event"})
outcome_events["Episode event"] = outcome_events["Episode event"].replace({
    "offer": "Makes an offer", "joins_offer": "Joins the offer", "founder_accepts": "Accepts the offer",
})
if not outcome_events.empty:
    display(Markdown("**What happened in the episode** (from the saved annotation record):"))
    display(HTML(outcome_events.to_html(index=False)))

display(Markdown("**What contributed to this prediction**"))
display(contributions.head(12))

<a id="across-the-100-videos"></a>
## 4. Across the full 100 episodes

### Summary statistics: what do the annotations and outcomes look like?

The outcome and turn summaries describe the **whole episode**, including negotiations and decisions. The family summaries below distinguish these full-record annotations from the model concepts, which use eligible interactions before the **first** investor decision.

The outcome is whether **any shark made an offer**, whether or not the founder accepted it. Turn and answer distributions pool the recorded turns across episodes, so episodes with more turns contribute more observations; pitch ratings and outcomes count each episode once.

In [ ]:
from analysis import corpus_overview as corpus

code_counts = px.code_value_counts(
    REFERENCE_RUN, instrument.books, manifest["video_ids"], pass_name="analysis"
)
# Exchanges are not used by the model and are excluded from this overview.
code_counts = code_counts[~code_counts.code.str.startswith("EX-")].copy()
episode_statistics = corpus.episode_summary(data)
pass_counts = data.videos.source.value_counts()
display(Markdown(
    f"**{len(data.videos):,} episodes · {len(data.turns):,} turns · "
    f"{(data.turns.function == 'question').sum():,} questions · "
    f"{(data.turns.function == 'answer').sum():,} answers · "
    f"{len(data.decisions):,} decision events.** "
    "Selected passes: " + ", ".join(f"{n} {name}" for name, n in pass_counts.items()) + "."
))
fig = corpus.overview_figure(code_counts, features)
display(fig)
plt.close(fig)
# A turn can contain more than one decision event; these are distinct counts.
display(HTML(
    '<details><summary><b>Episode lengths, interaction counts, and the investment ask</b></summary>'
    '<p>Each row summarizes episodes with available data. SD is the sample standard deviation; '
    'the 25th and 75th percentiles delimit the middle half of episodes. '
    'The ask (VC-ASK) is shown as dollars and equity percentage.</p>'
    + episode_statistics.to_html(index=False, float_format=lambda x: f"{x:,.1f}", na_rep="—")
    + '</details>'
))

### Distributions of the concept families 

The rating bars first show the whole-episode annotations for pitch preparedness and positivity, question severity, and answer quality. Grey hatched segments are missing ratings and stay in the denominator. Question severity runs from 1 to 3; the other ratings run from 1 to 5.

Below the figure, you can view distributions for each family and summary statistics of **all the concepts used by the model**, 

In [ ]:
fig = corpus.ratings_figure(code_counts)
display(fig)
plt.close(fig)
display(HTML(corpus.distribution_details(code_counts, features)))
# Exact counts and shares remain available as a DataFrame: code_counts.


### Human validation on a sample of 10-episode from the corpus

Two human coders independently reviewed **10 sampled episodes from the 100-episode corpus** using the same codebook Gemini received. The table reports Gwet’s AC1 or AC2, with Krippendorff’s α in parentheses and 95% intervals for AC1/AC2.


In [ ]:
from analysis import corpus_validation as validation

validation_dir = REFERENCE_RUN / "review/agreement"
validation_report = demo.read_json(validation_dir / "validation_table.json")
validation_agreement = validation.agreement_table(validation_report)
display(validation.styled_agreement(validation_agreement))


† **Small sample:** fewer than 20 distinct units; presentation and delivery estimates use only 10 episodes even when there are 20 Gemini–human judgments. The number of units varies because missing or off-scale ratings are excluded. 


The saved figure shows the same 12 assessed codes, with 95% intervals. Its dashed 0.61 line is a conventional reference, not a validation pass/fail rule.

In [ ]:
from IPython.display import Image as NotebookImage

display(NotebookImage(filename=str(validation_dir / "agreement_by_code.png"), width=1200))
validation_other_checks = validation.additional_checks(validation_report)
display(HTML(
    '<details><summary><b>Additional checks: the ask, decision timing, and paired delivery rankings</b></summary>'
    '<p>These use exact, same-turn, or ranking agreement, rather than AC1/AC2 or alpha. '
    'For first-decision timing, judgments that cannot be assigned to a reference turn are excluded: '
    '10 Gemini–human and 5 human–human judgments are undetermined in this sample. '
    'The ask uses the saved rule: amounts within 1% and identical equity. '
    'Pairwise ranking p-values are exact binomial tests; shared clips make the independence assumption optimistic.</p>'
    + validation_other_checks.to_html(index=False, escape=True)
    + '</details>'
))

### What concepts can tell us more about sharks' decisions?

The baseline model is to guess with random chance (50%). Then each family of concepts enter the model one at a time so we know exactly how much they contribute to the model's performance. First includes rating on the delivery and preparedness of the pitch and control variables (the type of venture, the asked investment, and the amount of interaction before the first decision). Then, the investors’ questions (how many, how hard, how framed); the entrepreneurs’ answers to sharks’ questions; the panel’s reactions to answers; and rapport and repair between entrepreneurs and sharks. The final model combines all four groups.

Panel A compares the pitch and the answers for episodes with and without an offer. Points show group means and lines show 95% bootstrap intervals; ratings use a 1–5 scale, while checkable specifics and admitted uncertainty are shown as percentages of answers.

In Panel B, the black line marks chance. The table and Panel B report performance above chance, using the saved held-out predictions for all 100 videos. Intervals come from the pitch bootstrap; p-values come from the existing label-permutation tests. They test prediction under shuffled outcomes, not the gain over the first model. The saved fitted results already contain these quantities.

Source: [chance comparisons and permutation tests](analysis/panel_bottleneck.py), [shared table and figure](analysis/proposal_figures.py).


In [ ]:
# Regenerate the paper Figure 2 with the shared grouped-means Panel A.
display(chance_table(saved["metrics"]))
fig = figure2(REFERENCE_RUN / "analysis/all100", tables=saved)
display(fig)
plt.close(fig)